# Phase 7 / Architecture 2: LegalSeq2Seq

In [1]:
import os, sys, json, time, pickle, hashlib, subprocess
from pathlib import Path
SESSION_START = time.time()
SESSION_END = SESSION_START + 3.5*3600 
BASE = Path('/kaggle/input/datasets/abdakt/a2-p3')
EVAL_WINDOWS = 8 
OUT = Path('/kaggle/working/p7_v2_a2')
OUT.mkdir(parents=True, exist_ok=True)
METHODS = ['full', 'lorma']
import torch

## LegalSeq2Seq encoder–decoder model and retrieval worker

In [2]:
# Architecture 2: LegalSeq2Seq — RoPE, RMSNorm, SwiGLU, cross-attention

import os, json, math, time, random, pickle, copy, zipfile, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

ARCH=2
import math


import torch


import torch.nn as nn


import torch.nn.functional as F


from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        z = x.float()
        z = z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-5)
        return z.to(x.dtype) * self.weight


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        self.heads = cfg['heads']
        self.head_dim = cfg['dim'] // self.heads
        self.causal = causal
        self.qkv = nn.Linear(cfg['dim'], 3 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        inv = 10000 ** (-torch.arange(0, self.head_dim, 2).float() / self.head_dim)
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos, sin = phase.cos()[None, None].to(x.dtype), phase.sin()[None, None].to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)

    def forward(self, x, mask=None, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        old = 0 if past is None else past[0].size(2)
        positions = torch.arange(old, old + t, device=x.device)
        q, k = self.rope(q, positions), self.rope(k, positions)
        if past is not None:
            k, v = torch.cat((past[0], k), 2), torch.cat((past[1], v), 2)
        allowed = None if mask is None else mask[:, None, None, :]
        fast_causal = self.causal and past is None and mask is None
        if self.causal and not fast_causal:
            causal_mask = torch.arange(k.size(2), device=x.device)[None, :] <= positions[:, None]
            allowed = causal_mask[None, None] if allowed is None else allowed & causal_mask[None, None]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                          is_causal=fast_causal, dropout_p=0.0)
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, d))
        return y, (k, v) if use_cache else None


class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads, self.head_dim = cfg['heads'], cfg['dim'] // cfg['heads']
        self.q = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        self.kv = nn.Linear(cfg['dim'], 2 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)

    def project_memory(self, memory):
        b, s, _ = memory.shape
        k, v = self.kv(memory).view(b, s, 2, self.heads, self.head_dim).unbind(2)
        return k.transpose(1, 2), v.transpose(1, 2)

    def forward(self, x, memory, mask, cached=None):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)
        k, v = self.project_memory(memory) if cached is None else cached
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, None, :], dropout_p=0.0)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, d))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.attn = SelfAttention(cfg, causal=False)
        self.ffn = FeedForward(cfg)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)[0]
        return x + self.ffn(self.norm2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = (RMSNorm(cfg['dim']) for _ in range(3))
        self.attn = SelfAttention(cfg, causal=True)
        self.cross = CrossAttention(cfg)
        self.ffn = FeedForward(cfg)

    def forward(self, x, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        a, cache = self.attn(self.norm1(x), target_mask, past, use_cache)
        x = x + a
        x = x + self.cross(self.norm2(x), memory, source_mask, cross_kv)
        return x + self.ffn(self.norm3(x)), cache


class LegalSeq2Seq(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg['encoder_layers'])])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg['decoder_layers'])])
        self.encoder_norm, self.decoder_norm = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.apply(self._init)
        for block in self.encoder:
            for weight in (block.attn.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(2 * len(self.encoder)))
        for block in self.decoder:
            for weight in (block.attn.out.weight, block.cross.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(3 * len(self.decoder)))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def encode(self, source, source_mask):
        x = self.embed(source)
        for block in self.encoder:
            if self.training and self.cfg['gradient_checkpointing']:
                x = checkpoint(block, x, source_mask, use_reentrant=False)
            else:
                x = block(x, source_mask)
        return self.encoder_norm(x)

    def decode(self, target, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        x, caches = self.embed(target), []
        for i, block in enumerate(self.decoder):
            if self.training and self.cfg['gradient_checkpointing']:
                def layer(z, mem, module=block):
                    return module(z, mem, source_mask, target_mask)[0]
                x = checkpoint(layer, x, memory, use_reentrant=False)
            else:
                x, cache = block(x, memory, source_mask, target_mask,
                                 None if past is None else past[i],
                                 None if cross_kv is None else cross_kv[i], use_cache)
                caches.append(cache)
        return self.decoder_norm(x), caches

    def forward(self, source, target, labels=None):
        source_mask = source.ne(self.cfg['pad_id'])
        memory = self.encode(source, source_mask)
        x, _ = self.decode(target, memory, source_mask, target.ne(self.cfg['pad_id']))
        if labels is None:
            return F.linear(x, self.embed.weight)
        h, y = x.flatten(0, 1), labels.flatten()
        total = h.new_zeros((), dtype=torch.float32)
        for start in range(0, len(y), 256):
            def part(z, truth):
                return F.cross_entropy(F.linear(z, self.embed.weight).float(), truth,
                                       ignore_index=-100, reduction='sum')
            if self.training:
                total = total + checkpoint(part, h[start:start+256], y[start:start+256], use_reentrant=False)
            else:
                total = total + part(h[start:start+256], y[start:start+256])
        return total



MODEL_CLASS=LegalSeq2Seq
class BudgetExpired(Exception): pass

def atomic_save(obj,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); torch.save(obj,tmp); os.replace(tmp,path)

def chunks(ids,size=512,cap=4):
    ids=ids or [1]
    if len(ids)<=size*cap: return [ids[i:i+size] for i in range(0,len(ids),size)]
    return [ids[i*(len(ids)-size)//(cap-1):i*(len(ids)-size)//(cap-1)+size] for i in range(cap)]

class BM25:
    def __init__(self,texts):
        counts=[Counter(self.words(t)) for t in texts]
        lengths=np.array([sum(c.values()) for c in counts]); avg=max(lengths.mean(),1)
        postings=defaultdict(list)
        for j,c in enumerate(counts):
            for w,tf in c.items(): postings[w].append((j,tf))
        self.n=len(texts); self.index={}
        for w,rows in postings.items():
            j,tf=map(np.array,zip(*rows)); df=len(j)
            idf=np.log1p((self.n-df+.5)/(df+.5))
            weight=idf*tf*2.5/(tf+1.5*(.25+.75*lengths[j]/avg))
            self.index[w]=(j,weight.astype('float32'))
    @staticmethod
    def words(text): return re.findall(r'[a-z0-9]+',text.lower())
    def rank(self,text):
        score=np.zeros(self.n,dtype='float32')
        for w in set(self.words(text)):
            if w in self.index:
                j,v=self.index[w]; score[j]+=v
        return np.argsort(-score,kind='stable').tolist()

def metric(order,gold):
    hits=sum(len(set(r[:7])&set(g)) for r,g in zip(order,gold))
    pred=7*len(gold); rel=sum(map(len,gold))
    return dict(f1=2*hits/max(1,pred+rel),precision=hits/max(1,pred),recall=hits/max(1,rel))

def fusion(a,b,weight=.5):
    out=[]
    for ra,rb in zip(a,b):
        scores=defaultdict(float)
        for r,j in enumerate(ra): scores[j]+=(1-weight)/(60+r+1)
        for r,j in enumerate(rb): scores[j]+=weight/(60+r+1)
        out.append(sorted(scores,key=lambda j:(-scores[j],j)))
    return out


ALPHAS=[round(float(a),2) for a in np.arange(.3,1.0001,.05)]   # dense weight; floor .3 so the Phase-1 model always contributes

def zrow(S):
    S=np.asarray(S,dtype=np.float32); return (S-S.mean(1,keepdims=True))/(S.std(1,keepdims=True)+1e-6)

def neutral(S,qid,cid):
    S=np.array(S,dtype=np.float32); index={k:j for j,k in enumerate(cid)}; n=S.shape[1]
    for i,k in enumerate(qid):
        j=index.get(k)
        if j is not None and n>1: S[i,j]=(S[i].sum()-S[i,j])/(n-1)
    return S

def order_of(S,qid,cid):
    S=np.array(S,dtype=np.float32); index={k:j for j,k in enumerate(cid)}
    for i,k in enumerate(qid):
        j=index.get(k)
        if j is not None: S[i,j]=-np.inf
    return np.argsort(-S,axis=1,kind='stable').tolist()

def fuse(dense,lex,a,qid,cid):
    if a>=1.0: return np.asarray(dense,dtype=np.float32)
    return a*zrow(neutral(dense,qid,cid))+(1-a)*zrow(lex)

class LoRMAPlus(nn.Module):
    def __init__(self,base,rank=16):
        super().__init__(); self.base=base
        self.a=nn.Linear(base.out_features,rank,bias=False)
        self.b=nn.Linear(rank,base.out_features,bias=False)
        nn.init.zeros_(self.b.weight)
    def forward(self,x):
        y=self.base(x); return y+self.b(self.a(y))

class Retriever(nn.Module):
    def __init__(self,cfg,method,weights=None):
        super().__init__(); self.cfg=dict(cfg); self.cfg['gradient_checkpointing']=False
        self.base=MODEL_CLASS(self.cfg)
        if weights is not None: self.base.load_state_dict(weights,strict=True)
        if ARCH==2: del self.base.decoder,self.base.decoder_norm
        d=cfg.get('dim',cfg.get('width')); self.proj=nn.Linear(d,256,bias=False)
        nn.init.orthogonal_(self.proj.weight)
        self.register_buffer('center',torch.zeros(d))
        self.register_buffer('scale',torch.ones(d))
        layers=self.layers()
        if method in ('partial','lorma'):
            self.base.requires_grad_(False)
            if method=='partial':
                for layer in list(layers)[-max(1,len(layers)//3):]: layer.requires_grad_(True)
            else:
                for layer in layers:
                    layer.attn.qkv=LoRMAPlus(layer.attn.qkv)
                    layer.attn.out=LoRMAPlus(layer.attn.out)
        if method=='distill':
            selected=list(layers)[1::2]
            if len(layers)%2: selected.append(layers[-1])
            if ARCH==2: self.base.encoder=nn.ModuleList(selected)
            else: self.base.blocks=nn.ModuleList(selected)
    def layers(self): return self.base.encoder if ARCH==2 else self.base.blocks
    def hidden(self,ids,mask):
        b=self.base
        if ARCH==2:
            x=b.embed(ids)
            for layer in b.encoder: x=checkpoint(layer,x,mask,use_reentrant=False) if self.training else layer(x,mask)
            return b.encoder_norm(x)
        if ARCH==3: return b.hidden(ids,mask.sum(1))
        x=b.embed(ids) if ARCH==1 else b.emb(ids)
        for layer in b.blocks:
            def step(z,module=layer):
                if ARCH==1: return module(z,torch.arange(ids.size(1),device=ids.device)[None],mask)[0]
                return module(z)[0]
            x=checkpoint(step,x,use_reentrant=False) if self.training else step(x)
        return b.norm(x)
    def pool(self,doc):
        dev=self.proj.weight.device; pooled=[]
        for chunk in doc:
            ids=torch.tensor(chunk,device=dev,dtype=torch.long)[None]
            mask=torch.ones_like(ids,dtype=torch.bool)
            x=self.hidden(ids,mask)
            pooled.append(x.float().mean(1))
        return torch.stack(pooled).mean(0)
    def forward(self,doc):
        return F.normalize(self.proj((self.pool(doc)-self.center)/self.scale).float(),dim=-1)

def candidate_batch(split,hard,group,rng):
    chosen=[]
    for i in group:
        gold=split['gold'][i]; chosen.append(rng.choice(sorted(gold)))
        hard_pool=[j for j in hard[i][:30] if j not in gold and split['cid'][j]!=split['qid'][i]]
        chosen.extend(rng.sample(hard_pool,min(3,len(hard_pool))))
        other=[j for j in range(len(split['c'])) if j not in gold and split['cid'][j]!=split['qid'][i]]
        chosen.extend(rng.sample(other,min(1,len(other))))
    return list(dict.fromkeys(chosen))

def cached_step(model,qs,cs,pos,scaler,teacher=None):
    docs=qs+cs
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):
        vectors=torch.cat([model(doc) for doc in docs])
    leaf=vectors.detach().requires_grad_(True); nq=len(qs)
    logits=leaf[:nq]@leaf[nq:].T/.07
    loss=(logits.logsumexp(1)-logits.masked_fill(~pos,-torch.inf).logsumexp(1)).mean()
    if teacher is not None:
        with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):
            t=torch.cat([teacher(doc) for doc in docs]); target=(t[:nq]@t[nq:].T/.14).softmax(1)
        loss=.5*loss+2*F.kl_div((logits/2).log_softmax(1),target,reduction='batchmean')
    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite retrieval loss')
    gradients=torch.autograd.grad(loss,leaf)[0]
    for doc,g in zip(docs,gradients):
        with torch.autocast('cuda',dtype=torch.float16): z=model(doc)
        scaler.scale((z.float()*g).sum()).backward()
    return float(loss.detach())

@torch.inference_mode()
def dense_scores(model,split,deadline):
    model.eval(); vectors=[]
    for docs in (split['q'],split['c']):
        rows=[]
        for doc in docs:
            if time.time()>deadline: raise BudgetExpired()
            with torch.autocast('cuda',dtype=torch.float16): rows.append(model(doc).cpu())
        vectors.append(torch.cat(rows))
    return (vectors[0].float()@vectors[1].float().T).numpy()

def load_retriever(base,cfg,method,path=None):
    ck=torch.load(Path(base)/'best.pt',map_location='cpu',weights_only=False)
    model=Retriever(cfg,method,ck['model']); del ck
    if path: model.load_state_dict(torch.load(path,map_location='cpu',weights_only=False)['model'])
    return model

def export_predictions(folder,split,order,name):
    assert len(order)==len(split['qid'])
    pred={}
    for q,row in zip(split['qid'],order):
        top=[split['cid'][j] for j in row[:7]]
        assert len(top)==7 and len(set(top))==7
        pred[q]=top
    with zipfile.ZipFile(folder/f'{name}.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',json.dumps(pred))

def worker(config_path,method,gpu,train_end,deadline,teacher_path=None):
    inference_only = train_end <= time.time()
    torch.set_num_threads(2); torch.cuda.set_device(gpu)
    torch.manual_seed(781); torch.cuda.manual_seed(781)
    cfg_run=json.loads(Path(config_path).read_text()); base=cfg_run['base']; out=Path(cfg_run['out'])
    data=pickle.load(open(out/'data.pkl','rb')); cfg=cfg_run['model']
    folder=out/method; folder.mkdir(exist_ok=True)
    model=load_retriever(base,cfg,method).cuda(); teacher=None
    if method=='distill':
        teacher=load_retriever(base,cfg,'full',teacher_path).cuda().eval().requires_grad_(False)
        keep=list(teacher.layers())[1::2]
        if len(teacher.layers())%2: keep.append(teacher.layers()[-1])
        source=teacher.state_dict(); target=model.state_dict(); prefix='base.encoder.' if ARCH==2 else 'base.blocks.'
        model.load_state_dict({k:source[k] if not k.startswith(prefix) else v for k,v in target.items()})
        for dst,src in zip(model.layers(),keep): dst.load_state_dict(src.state_dict())
    if method!='distill':
        model.eval()
        choices=random.Random(781).sample(data['train']['c'],min(192,len(data['train']['c'])))
        with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):
            features=torch.cat([model.pool(doc) for doc in choices])
            model.center.copy_(features.mean(0))
            sd=features.std(0,unbiased=False)
            model.scale.copy_(sd.clamp_min(sd.median()*.1+1e-6))
    lr={'full':2e-5,'partial':5e-5,'lorma':3e-4,'distill':1e-5}[method]
    opt=torch.optim.AdamW([{'params':[p for p in model.base.parameters() if p.requires_grad],'initial_lr':lr},
                           {'params':model.proj.parameters(),'initial_lr':5e-4}],lr=lr,weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda'); rng=random.Random(781)
    step=0; best=-1.; stale=0; epoch=0; offset=0; rows=[]
    last=folder/'last.pt'
    if last.exists():
        ck=torch.load(last,map_location='cpu',weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
        step,best,stale,epoch,offset,rows=[ck[k] for k in ('step','best','stale','epoch','offset','rows')]
        rng.setstate(ck['rng']); del ck
    def save():
        atomic_save(dict(model=model.state_dict(),optimizer=opt.state_dict(),scaler=scaler.state_dict(),
            step=step,best=best,stale=stale,epoch=epoch,offset=offset,rows=rows,rng=rng.getstate(),config=cfg_run),last)
    eval_seconds=600.; update_seconds=60.; last_eval_step=-1
    def evaluate():
        nonlocal best,stale,eval_seconds,last_eval_step
        start=time.time(); S=dense_scores(model,data['dev'],deadline-300); order=order_of(S,data['dev']['qid'],data['dev']['cid'])
        val=metric(order,data['dev']['gold']); eval_seconds=time.time()-start; last_eval_step=step
        print(method,'epoch',epoch,'step',step,val,'eval_seconds',round(eval_seconds),flush=True)
        with (folder/'history.jsonl').open('a') as f: f.write(json.dumps(dict(epoch=epoch,step=step,eval_seconds=eval_seconds,**val))+'\n')
        if val['f1']>best:
            best=val['f1']; stale=0
            atomic_save(dict(model=model.state_dict(),method=method,config=cfg_run),folder/'best.pt')
            np.save(folder/'dev_scores.npy',S)
        else: stale+=1
        save(); model.train()
    if best<0 and not inference_only: evaluate() # Always preserve an evaluated baseline and recoverable checkpoint.
    consecutive_overflows=0
    while time.time()+max(120,1.5*update_seconds)+1.3*eval_seconds<train_end and epoch<12 and stale<3:
        if not rows or offset>=len(rows):
            rows=[i for i,g in enumerate(data['train']['gold']) if g]; rng.shuffle(rows); offset=0; epoch+=1
        model.train(); start=time.time(); group=rows[offset:offset+4]
        sampler_state=rng.getstate()
        cand=candidate_batch(data['train'],data['hard'],group,rng)
        qs=[data['train']['q'][i] for i in group]; cs=[data['train']['c'][j] for j in cand]
        pos=torch.tensor([[j in data['train']['gold'][i] for j in cand] for i in group],device='cuda')
        opt.zero_grad(set_to_none=True)
        for pg in opt.param_groups: pg['lr']=pg['initial_lr']*min(1.,(step+1)/20)
        loss=cached_step(model,qs,cs,pos,scaler,teacher)
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)
        before=scaler.get_scale(); scaler.step(opt); scaler.update()
        if scaler.get_scale()<before:
            consecutive_overflows+=1
            rng.setstate(sampler_state)
            print(method,'FP16 overflow; retrying same batch at scale',scaler.get_scale(),flush=True)
            if consecutive_overflows>=12:
                raise FloatingPointError('12 successive skipped updates on one batch')
            continue
        consecutive_overflows=0
        offset+=len(group); step+=1; update_seconds=.8*update_seconds+.2*(time.time()-start)
        if step%10==0: print(method,'step',step,'loss',round(loss,4),'seconds/update',round(update_seconds,1),flush=True)
        if step%25==0: save()
        if offset>=len(rows): evaluate()
    if not inference_only and step!=last_eval_step and time.time()+1.3*eval_seconds<deadline-300: evaluate()
    if not inference_only: save()
    if not (folder/'best.pt').exists():
        print(method,'no complete dev evaluation within budget; checkpoint saved but no selected model',flush=True)
        return
    del opt,model; torch.cuda.empty_cache()
    if teacher is not None: del teacher; torch.cuda.empty_cache()
    model=load_retriever(base,cfg,method,folder/'best.pt').cuda().eval()
    D=data['dev']; T=data['test']; gold=D['gold']; Sd=np.load(folder/'dev_scores.npy'); Ld=data['lexs']['dev']
    options=[(a,metric(order_of(fuse(Sd,Ld,a,D['qid'],D['cid']),D['qid'],D['cid']),gold)['f1']) for a in ALPHAS]
    alpha,score=max(options,key=lambda z:(z[1],z[0]))
    dense_f1=metric(order_of(Sd,D['qid'],D['cid']),gold)['f1']
    (folder/'selection.json').write_text(json.dumps(dict(dense_weight=alpha,lexical_weight=round(1-alpha,2),dev_f1_selected=score,dev_f1_dense_only=dense_f1,lexical=data['lexinfo'],options=options)))
    print(method,'dense-only dev F1',round(dense_f1,4),'| selected alpha',alpha,'dev F1',round(score,4),flush=True)
    St=dense_scores(model,T,deadline-60); np.save(folder/'test_scores.npy',St)
    export_predictions(folder,T,order_of(St,T['qid'],T['cid']),'submission_neural')
    export_predictions(folder,T,order_of(fuse(St,data['lexs']['test'],alpha,T['qid'],T['cid']),T['qid'],T['cid']),'submission_selected')
    print(method,'finished',flush=True)

## Data preparation and TF-IDF hard negatives

In [3]:
from tokenizers import Tokenizer
from huggingface_hub import HfApi, hf_hub_download
secret=os.environ.get('HF_TOKEN')
repo='Exploration-Lab/CS-781-Capstone'
revision=HfApi(token=secret).dataset_info(repo).sha
ck=torch.load(BASE/'best.pt',map_location='cpu',weights_only=False)
model_cfg=json.loads((BASE/'config.json').read_text()); model_cfg.update(ck.get('config',{})); del ck
tok=Tokenizer.from_file(str(BASE/'tokenizer.json')); tok.no_truncation(); tok.no_padding()
vocab=model_cfg.get('vocab_size',model_cfg.get('vocab'))
run=dict(base=str(BASE),out=str(OUT),model=model_cfg,methods=METHODS,revision=revision,
         representation=f'train head-tail 2x512; dev/test spread {EVAL_WINDOWS}x512; standardized retrieval features; TF-IDF(1-2gram) lexical fusion',
         negatives=dict(hard=3,random=1),queries_per_update=4,seed=781,
         tokenizer_sha256=hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest(),
         eval_windows=EVAL_WINDOWS,lexical='tf_uni/tf_stop/tf_bi dev-selected, z-score fusion alpha>=.3',pipeline_version='train_fusion_v2')
(OUT/'run.json').write_text(json.dumps(run,indent=2))
def records(raw):
    if isinstance(raw,dict): return [(str(k),v) for k,v in raw.items()]
    return [(str(next(r[k] for k in ('id','case_id','query_id','candidate_id','document_id') if k in r)),r) for r in raw]
def read(split,suffix):
    p=hf_hub_download(repo,f'Phase_7_PCR/{split.title()}/{split}{suffix}.json',repo_type='dataset',revision=revision,token=secret)
    return records(json.loads(Path(p).read_text()))
def text(row):
    x=row['text'] if isinstance(row,dict) else row
    return x if isinstance(x,str) else ' '.join(x)
if not (OUT/'data.pkl').exists():
    from sklearn.feature_extraction.text import TfidfVectorizer
    LEX_CFGS={'tf_uni':dict(sublinear_tf=True,min_df=2,max_df=.6),
              'tf_stop':dict(sublinear_tf=True,min_df=2,max_df=.5,stop_words='english'),
              'tf_bi':dict(sublinear_tf=True,min_df=2,max_df=.5,stop_words='english',ngram_range=(1,2),max_features=400000)}
    def lex_matrices(qt,ct):
        out={}
        for name,kw in LEX_CFGS.items():
            vec=TfidfVectorizer(dtype=np.float32,**kw); cm=vec.fit_transform(ct); out[name]=(vec.transform(qt)@cm.T).toarray()
        return out
    data={}; coverage={}; raw={}
    for split in ['train','dev','test']:
        q,c=read(split,''),read(split,'_candidates')
        cid=[k for k,_ in c]; qid=[k for k,_ in q]; index={k:i for i,k in enumerate(cid)}
        qt,ct=[text(r) for _,r in q],[text(r) for _,r in c]
        gold=[{index[str(k)] for k in r['relevant_candidates']} for _,r in q] if split!='test' else []
        if split=='train':
            vec=TfidfVectorizer(dtype=np.float32,**LEX_CFGS['tf_bi']); cm=vec.fit_transform(ct)
            S=(vec.transform(qt)@cm.T).toarray()
            for i,k in enumerate(qid):
                if k in index: S[i,index[k]]=-1e9
            lexical=np.argsort(-S,axis=1,kind='stable')[:,:400].tolist(); del S,vec,cm
        else:
            raw[split]=lex_matrices(qt,ct)
        encoded=[]
        for kind,texts in [('q',qt),('c',ct)]:
            docs=[]; original=retained=0
            for t in texts:
                ids=tok.encode(t,add_special_tokens=False).ids
                windows=chunks(ids,size=512,cap=2 if split=='train' else EVAL_WINDOWS); docs.append(windows)
                original+=len(ids); retained+=min(len(ids),sum(map(len,windows)))
            encoded.append(docs); coverage[split+'_'+kind]=dict(documents=len(docs),tokens=original,retained=retained)
        data[split]=dict(qid=qid,cid=cid,q=encoded[0],c=encoded[1],gold=gold)
        if split=='train': data['hard']=[[j for j in row if j not in g][:100] for row,g in zip(lexical,gold)]
        print(split,len(q),len(c),flush=True)
    D=data['dev']; neu={sp:{k:neutral(v,data[sp]['qid'],data[sp]['cid']) for k,v in raw[sp].items()} for sp in raw}
    single={k:metric(order_of(v,D['qid'],D['cid']),D['gold'])['f1'] for k,v in neu['dev'].items()}
    for k,v in single.items(): print(f'  lexical {k:7s} dev F1@7 {v:.4f}')
    ranked=sorted(single,key=lambda k:-single[k]); choices={k:[k] for k in ranked}; choices['mix_top2']=ranked[:2]; choices['mix_all']=ranked
    mixed=lambda sp,ks: np.mean([zrow(neu[sp][k]) for k in ks],0)
    scores={n:metric(order_of(mixed('dev',ks),D['qid'],D['cid']),D['gold'])['f1'] for n,ks in choices.items()}
    for n,v in scores.items():
        if n.startswith('mix'): print(f'  lexical {n:8s} dev F1@7 {v:.4f}')
    best_name=max(scores,key=scores.get)
    data['lexs']={sp:mixed(sp,choices[best_name]).astype(np.float32) for sp in ('dev','test')}
    data['lexinfo']=dict(chosen=best_name,members=choices[best_name],dev_f1=scores[best_name],singles=single)
    with (OUT/'data.pkl.tmp').open('wb') as f: pickle.dump(data,f)
    os.replace(OUT/'data.pkl.tmp',OUT/'data.pkl')
    (OUT/'coverage.json').write_text(json.dumps(coverage,indent=2))
else:
    data=pickle.load(open(OUT/'data.pkl','rb'))
print('LEXICAL BASELINE:',data['lexinfo'])
print('Preparation elapsed minutes:',round((time.time()-SESSION_START)/60,1))

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train 827 4320


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (3 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test 237 1727
  lexical tf_uni  dev F1@7 0.3546
  lexical tf_stop dev F1@7 0.3602
  lexical tf_bi   dev F1@7 0.4314
  lexical mix_top2 dev F1@7 0.4136
  lexical mix_all  dev F1@7 0.3969
LEXICAL BASELINE: {'chosen': 'tf_bi', 'members': ['tf_bi'], 'dev_f1': 0.4313507504168983, 'singles': {'tf_uni': 0.3546414674819344, 'tf_stop': 0.3602001111728738, 'tf_bi': 0.4313507504168983}}
Preparation elapsed minutes: 5.2


## Training

In [4]:
import cloudpickle
def launch_jobs(jobs):
    processes=[]
    for method,gpu,train_end,deadline,teacher in jobs:
        payload=OUT/f'_worker_{method}.pkl'
        with payload.open('wb') as f:
            cloudpickle.dump((worker,(str(OUT/'run.json'),method,gpu,train_end,deadline,teacher)),f)
        program="import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
        processes.append((method,subprocess.Popen([sys.executable,'-u','-c',program,str(payload)])))
    while any(p.poll() is None for _,p in processes):
        if time.time()>SESSION_END-30:
            raise TimeoutError('Session cutoff: use the saved checkpoints/output files.')
        for method,p in processes:
            if p.poll() not in (None,0): raise RuntimeError(f'{method} worker exited with {p.returncode}; see traceback above.')
        time.sleep(2)
    for _,p in processes:
        if p.poll() is None: p.terminate()
    for _,p in processes: p.wait()
    for method,_ in processes: (OUT/f'_worker_{method}.pkl').unlink(missing_ok=True)
H=3600  # 4-hour session: prep ~15 min, training until ~3h15, then final dev scoring + test inference (~20 min) with a safety buffer.
if METHODS==['full','distill']:
    launch_jobs([('full',0,SESSION_START+int(1.6*H),SESSION_START+2*H,None)])
    assert (OUT/'full'/'best.pt').exists(), 'Teacher did not produce an evaluated checkpoint.'
    launch_jobs([('distill',1,SESSION_START+int(3.4*H),SESSION_END-120,OUT/'full'/'best.pt')])
else:
    launch_jobs([(m,g,SESSION_START+int(3.25*H),SESSION_END-120,None) for g,m in enumerate(METHODS)])

full epoch 0 step 0 {'f1': 0.2045580878265703, 'precision': 0.22276029055690072, 'recall': 0.18910585817060638} eval_seconds 70
full FP16 overflow; retrying same batch at scale 32768.0
lorma epoch 0 step 0 {'f1': 0.2045580878265703, 'precision': 0.22276029055690072, 'recall': 0.18910585817060638} eval_seconds 86
full step 10 loss 2.1842 seconds/update 9.3
lorma step 10 loss 2.4286 seconds/update 9.6
full step 20 loss 1.6116 seconds/update 3.9
lorma step 20 loss 1.6333 seconds/update 4.2
full step 30 loss 2.0443 seconds/update 3.3
lorma step 30 loss 2.1327 seconds/update 3.6
full step 40 loss 2.3749 seconds/update 3.2
lorma step 40 loss 2.6978 seconds/update 3.5
full step 50 loss 2.9048 seconds/update 3.2
lorma step 50 loss 2.647 seconds/update 3.5
full step 60 loss 2.5154 seconds/update 3.2
lorma step 60 loss 2.671 seconds/update 3.5
full step 70 loss 3.4558 seconds/update 3.2
lorma step 70 loss 4.5954 seconds/update 3.5
full step 80 loss 1.5888 seconds/update 3.3
full step 90 loss 1.9

## Submissions

In [5]:
from IPython.display import display, FileLink
for method in METHODS:
    p=OUT/method/'selection.json'
    if p.exists(): d=json.loads(p.read_text()); print(method,{k:d[k] for k in ('dense_weight','lexical_weight','dev_f1_dense_only','dev_f1_selected')})
    for p in sorted((OUT/method).glob('submission_*.zip')): display(FileLink(str(p)))
print('Preserve the complete output directory, including run.json, data.pkl, best.pt and last.pt.')

full {'dense_weight': 0.35, 'lexical_weight': 0.65, 'dev_f1_dense_only': 0.23346303501945526, 'dev_f1_selected': 0.43468593663146193}


/kaggle/working/p7_v2_a2/full/submission_neural.zip

/kaggle/working/p7_v2_a2/full/submission_selected.zip

lorma {'dense_weight': 0.3, 'lexical_weight': 0.7, 'dev_f1_dense_only': 0.23235130628126738, 'dev_f1_selected': 0.4313507504168983}


/kaggle/working/p7_v2_a2/lorma/submission_neural.zip

/kaggle/working/p7_v2_a2/lorma/submission_selected.zip

Preserve the complete output directory, including run.json, data.pkl, best.pt and last.pt.
